# IT-Dienstleister: Einfache Anomalie-Erkennung in Server-Logs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_10/it_dienstleister_simple.ipynb)

**Lernziel:** Eine einfache einseitige 3σ-Regel auf einer historischen Referenz kalibrieren, spätere Messwerte prüfen und Fehlalarme von injizierten Störungen unterscheiden.

**Vereinfachtes Szenario:** Überwachung der Server-Response-Times und Erkennung von Performance-Problemen

**Lehrbeispiel!** Die Daten sind künstlich erzeugt und stehen in keinem Bezug zu einem realen Unternehmen. Wir simulieren normalverteilte Antwortzeiten mit einzelnen starken Verzögerungen. Reale Antwortzeiten sind häufig rechtsschief und zeitlich abhängig.

Jede Zeile ist **ein Messwert pro Minute**, nicht die Gesamtheit aller Requests. Die ersten vier Stunden dienen als zuvor freigegebene störungsfreie Referenz. Die anschließenden 20 Stunden werden mit einer eingefrorenen Schwelle überwacht. Ohne eine geeignete historische Referenz wäre dieses Vorgehen nicht gerechtfertigt.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Simuliere Server-Response-Times über 24 Stunden
np.random.seed(42)
n_minutes = 24 * 60  # 24 Stunden in Minuten

# Zeitstempel erstellen
start_time = datetime(2026, 1, 1)
timestamps = [start_time + timedelta(minutes=i) for i in range(n_minutes)]

# Normale Response Times (normalverteilt um 200ms)
normal_response = np.random.normal(200, 30, n_minutes)

# Einige Anomalien hinzufügen (Server-Probleme)
anomaly_indices = [300, 301, 302, 800, 801, 1200, 1201, 1202, 1203]  # Problemphasen
response_times = normal_response.copy()
response_times[anomaly_indices] = np.random.normal(800, 100, len(anomaly_indices))  # Deutlich langsamer

df = pd.DataFrame({
    'timestamp': timestamps,
    'response_time_ms': response_times.round(1)
})

print(f"Server-Logs für {n_minutes/60:.0f} Stunden")
print("\nErste 5 Einträge:")
df.head()

## 1. Schwelle nur aus der Vergangenheit bestimmen

Wir schätzen Mittelwert und Stichproben-Standardabweichung aus den ersten 240 Minuten. **Spätere Werte und die bekannten Simulations-Anomalien werden nicht zur Kalibrierung verwendet.** Würden die hohen Störungswerte in die Referenz eingehen, erhöhten sie Mittelwert und Streuung und könnten die Erkennung abschwächen.

Für hohe Antwortzeiten verwenden wir die **einseitige obere Schwelle** `Mittelwert + 3 * Standardabweichung`.

Die bekannte 3σ-Regel gilt bei einer idealen Normalverteilung mit bekannten Parametern: Rund **99,73 % liegen zwischen μ−3σ und μ+3σ**, rund **99,865 % unter μ+3σ**. Das ist keine garantierte Quote für diese Stichprobe oder reale Serverdaten. Unsere Parameter sind geschätzt; außerdem ist die Gesamtverteilung mit injizierten Störungen keine einzelne Normalverteilung.

In [ ]:
baseline_df = df.iloc[:240].copy()
monitoring_df = df.iloc[240:].copy()
assert baseline_df['timestamp'].max() < monitoring_df['timestamp'].min()

mean_response = baseline_df['response_time_ms'].mean()
std_response = baseline_df['response_time_ms'].std()
threshold = mean_response + 3 * std_response

plt.figure(figsize=(12, 6))

bins = np.linspace(df['response_time_ms'].min(), df['response_time_ms'].max(), 51)
plt.hist(monitoring_df['response_time_ms'], bins=bins, alpha=0.5, color='lightblue',
         edgecolor='black', label='Spätere Messwerte (nicht zur Kalibrierung)')
plt.hist(baseline_df['response_time_ms'], bins=bins, alpha=0.7, color='gray',
         edgecolor='black', label='Historische Referenz: erste 4 Stunden')

# Statistische Marker
plt.axvline(mean_response, color='green', linestyle='-', linewidth=2, 
           label=f'Referenz-Mittelwert ({mean_response:.0f} ms)')
plt.axvline(threshold, color='red', linestyle='--', linewidth=2,
           label=f'Eingefrorene obere 3σ-Grenze ({threshold:.0f} ms)')

plt.xlabel('Response Time (ms)')
plt.ylabel('Häufigkeit (Anzahl Minutenmesswerte)')
plt.title('Historische Referenz und spätere Überwachung\nUnterschiedliche Intervalllängen: 240 bzw. 1200 Messwerte')
plt.legend()
plt.grid(True, alpha=0.3)

# Statistik-Box
stats_text = f'Referenz-Schätzwerte:\nMittelwert = {mean_response:.1f} ms\nStandardabweichung = {std_response:.1f} ms\nObere Schwelle = {threshold:.1f} ms'
plt.text(0.98, 0.65, stats_text, transform=plt.gca().transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
         fontsize=10)

plt.tight_layout()
plt.show()

print("📊 AUSGANGSLAGE:")
print(f"• Referenz-Mittelwert: {mean_response:.1f} ms; obere Alarmschwelle: {threshold:.1f} ms")
print("• Die Gesamtverteilung mit Störungen ist keine einzelne Normalverteilung.")
print("• Eine Überschreitung ist ein Alarmhinweis, kein Beweis für einen Serverfehler.")

In [ ]:
print("=== SERVER-MONITORING EINES IT-DIENSTLEISTERS ===")
print(f"Referenz-Mittelwert: {mean_response:.1f} ms")
print(f"Referenz-Standardabweichung: {std_response:.1f} ms")
print(f"Unveränderte Schwelle (Referenz-Mittelwert + 3s): {threshold:.1f} ms")

# Anomalien identifizieren
monitoring_df['is_anomaly'] = monitoring_df['response_time_ms'] > threshold
n_anomalies = int(monitoring_df['is_anomaly'].sum())

print(f"\n🚨 Anzahl Alarmhinweise im späteren Intervall: {n_anomalies}")
print(f"Alarmrate: {n_anomalies/len(monitoring_df)*100:.2f}%")

In [ ]:
# Visualisierung der Response Times und Anomalien
plt.figure(figsize=(15, 8))

plt.scatter(baseline_df['timestamp'], baseline_df['response_time_ms'],
           alpha=0.5, color='gray', s=15, label='Referenz (keine Überwachungsauswertung)')
plt.axvline(monitoring_df['timestamp'].iloc[0], color='gray', linestyle=':', label='Beginn der Überwachung')
normal_data = monitoring_df[~monitoring_df['is_anomaly']]
plt.scatter(normal_data['timestamp'], normal_data['response_time_ms'],
           alpha=0.6, color='blue', s=20, label='Ohne Alarm')

# Anomalien in rot
anomaly_data = monitoring_df[monitoring_df['is_anomaly']]
plt.scatter(anomaly_data['timestamp'], anomaly_data['response_time_ms'],
           alpha=0.8, color='red', s=50, label='Alarmhinweis', marker='x')

# Schwellwert als horizontale Linie
plt.axhline(y=threshold, color='orange', linestyle='--', linewidth=2, 
           label=f'Schwellwert ({threshold:.0f} ms)')

plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=2))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xlabel('Uhrzeit (ein Messwert pro Minute)')
plt.ylabel('Response Time (ms)')
plt.title('Server-Monitoring mit historisch festgelegter Schwelle')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Detaillierte Anomalie-Analyse
if n_anomalies > 0:
    print("=== ANOMALIE-DETAILS ===")
    anomaly_details = monitoring_df[monitoring_df['is_anomaly']][['timestamp', 'response_time_ms']].copy()
    anomaly_details['zeit'] = anomaly_details['timestamp'].dt.strftime('%H:%M')
    
    for _, row in anomaly_details.iterrows():
        print(f"⚠️  {row['zeit']} Uhr: {row['response_time_ms']:.1f} ms (Normal: ~{mean_response:.0f} ms)")
    
    print("\n=== SIMULIERTE ALERTS: KEIN REALER VERSAND ===")
    worst_response = monitoring_df['response_time_ms'].max()
    worst_time = monitoring_df.loc[monitoring_df['response_time_ms'].idxmax(), 'timestamp']
    
    print(f"🔴 KRITISCH: Langsamste Response um {worst_time.strftime('%H:%M')} Uhr: {worst_response:.1f} ms")
    print("📱 Demo: Hier könnte eine SMS an den On-Call Engineer ausgelöst werden. Es wurde nichts versendet.")
    print("📧 Demo: Hier könnte eine E-Mail an das IT-Team ausgelöst werden. Es wurde nichts versendet.")
else:
    print("Keine Schwellenüberschreitungen gefunden. Das beweist nicht, dass der Server fehlerfrei läuft.")

## 2. Erkennen wir die injizierten Störungen? Gibt es Fehlalarme?

Erst jetzt verwenden wir die bekannten Störungspositionen aus der Simulation zur Bewertung, nicht zur Schwellenwahl. Die Auswertung umfasst nur die 1200 späteren Messwerte.

**Precision** ist der Anteil injizierter Störungen unter den Alarmen; **Recall** der Anteil erkannter unter allen injizierten Störungen. Auch ein normaler Messwert kann die 3σ-Grenze zufällig überschreiten. Eine hohe Erkennungsrate schließt Fehlalarme nicht aus.

Die sehr starken simulierten Verzögerungen sind leicht erkennbar. Das Resultat belegt keine Eignung für echte Logs oder subtilere Störungen. Nachträgliches Optimieren der Schwelle auf diesen Ergebnissen würde ein weiteres unabhängiges Prüfintervall erfordern.

In [ ]:
true_anomaly = monitoring_df.index.isin(anomaly_indices)
predicted_anomaly = monitoring_df['is_anomaly'].to_numpy()
evaluation = pd.DataFrame(
    confusion_matrix(true_anomaly, predicted_anomaly, labels=[False, True]),
    index=['Normal', 'Injizierte Anomalie'],
    columns=['Kein Alarm', 'Alarm']
)
print('Zeilen: tatsächlicher Simulationszustand; Spalten: Detektorentscheidung')
print(evaluation)
print(f'Precision: {precision_score(true_anomaly, predicted_anomaly):.1%}')
print(f'Recall: {recall_score(true_anomaly, predicted_anomaly):.1%}')
print(f"Fehlalarme: {evaluation.loc['Normal', 'Alarm']}; übersehene injizierte Störungen: {evaluation.loc['Injizierte Anomalie', 'Kein Alarm']}")

## IT Operations Impact

**Business Value:**
- 🚨 **Alarmhinweise:** Auffällige Messwerte zur weiteren Untersuchung markieren; betroffene Requests sind bereits langsam
- ⚡ **Schnelle Reaktion:** Eine reale Monitoring-Integration könnte Benachrichtigungen auslösen; hier wird nur Text ausgegeben
- 📊 **Bewertung:** Erkannte Störungen und Fehlalarme gemeinsam betrachten
- 💰 Möglicher betrieblicher Nutzen muss anhand echter Daten und Alarmkosten geprüft werden

**Einfache Regel:** Antwortzeit > Referenz-Mittelwert + 3 × Referenz-Standardabweichung → Alarmhinweis, kein gesicherter Fehlernachweis.

**Grenzen:** Die Referenz muss geeignet sein; Tagesmuster, Lastwechsel und Drift können die feste Schwelle unbrauchbar machen. Niedrige Antwortzeiten und andere Fehlertypen werden mit dieser einseitigen Regel nicht untersucht.

**Einordnung:** Dies ist ein statistischer regelbasierter Detektor, kein trainierter Clustering-Algorithmus. Bei echten rechtsschiefen Antwortzeiten wären historische Quantile oder robuste Verfahren mögliche Alternativen, jeweils mit separater Kalibrierung und Bewertung auf späteren Daten.